<a href="https://colab.research.google.com/github/ParagWadhai/Sanskrit_English_Embedding_Finetune/blob/main/Sanskrit_English_Embedding_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sanskrit–English Multilingual Embedding Fine-Tuning
### Fine-tune a retrieval embedding model for cross-lingual Sanskrit ↔ English semantic search

---
**Pipeline Overview:**
1. Load Bhagavad Gita dataset (Sanskrit + English)
2. Build triplets (anchor, positive, hard negative)
3. Fine-tune `multilingual-e5-small` with MultipleNegativesRankingLoss
4. Build FAISS retrieval index
5. Evaluate with Recall@K, MRR, nDCG
6. Mini RAG demo

## Section 1: Install Dependencies

In [ ]:
%%capture
!pip install sentence-transformers datasets faiss-cpu indic-transliteration accelerate

In [ ]:
import os, random, json, warnings
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation,
    SentencesDataset,
)
from torch.utils.data import DataLoader
import faiss
from collections import defaultdict
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## Section 2: Load & Explore Dataset

In [ ]:
print("Loading Bhagavad Gita dataset...")
raw_dataset = load_dataset("JDhruv14/Bhagavad-Gita_Dataset")
print(raw_dataset)
print("\nSample entry:")
print(raw_dataset['train'][0])

In [ ]:
# ── Inspect columns ──────────────────────────────────────────────────────────
df = pd.DataFrame(raw_dataset['train'])
print("Columns:", df.columns.tolist())
print(f"\nTotal rows: {len(df)}")
df.head(3)

In [ ]:
# ── Flexible column detection ─────────────────────────────────────────────────
# Map whatever column names exist to our internal names
col_map = {}
cols_lower = {c.lower(): c for c in df.columns}

# Sanskrit / shloka
for candidate in ['sanskrit', 'shloka', 'verse', 'text_sanskrit', 'original']:
    if candidate in cols_lower:
        col_map['sanskrit'] = cols_lower[candidate]; break

# English translation
for candidate in ['english', 'translation', 'meaning', 'text_english', 'translation_english']:
    if candidate in cols_lower:
        col_map['english'] = cols_lower[candidate]; break

# Chapter
for candidate in ['chapter', 'chapter_number', 'adhyaya']:
    if candidate in cols_lower:
        col_map['chapter'] = cols_lower[candidate]; break

# Verse number
for candidate in ['verse', 'verse_number', 'shloka_number', 'sloka_number']:
    if candidate in cols_lower and candidate != col_map.get('sanskrit', '').lower():
        col_map['verse_num'] = cols_lower[candidate]; break

print("Column mapping:", col_map)
assert 'sanskrit' in col_map and 'english' in col_map, \
    f"Could not auto-detect Sanskrit/English columns. Please set col_map manually. Columns: {df.columns.tolist()}"

In [ ]:
# ── Build clean DataFrame ─────────────────────────────────────────────────────
clean = pd.DataFrame()
clean['sanskrit'] = df[col_map['sanskrit']].astype(str).str.strip()
clean['english']  = df[col_map['english']].astype(str).str.strip()
if 'chapter' in col_map:
    clean['chapter'] = df[col_map['chapter']]
if 'verse_num' in col_map:
    clean['verse_num'] = df[col_map['verse_num']]

# Drop empty rows
clean = clean[(clean['sanskrit'].str.len() > 5) & (clean['english'].str.len() > 5)].reset_index(drop=True)
print(f"Clean dataset size: {len(clean)} verses")
clean.head(3)

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
clean['english'].str.len().hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('English translation length (chars)')
axes[0].set_xlabel('Characters')

clean['sanskrit'].str.len().hist(bins=40, ax=axes[1], color='saddlebrown', edgecolor='white')
axes[1].set_title('Sanskrit verse length (chars)')
axes[1].set_xlabel('Characters')
plt.tight_layout()
plt.show()

if 'chapter' in clean.columns:
    print("\nVerses per chapter:")
    print(clean['chapter'].value_counts().sort_index())

## 🔗 Section 3: Build Training Triplets

We create **three types** of training pairs:

| Type | Anchor | Positive | Strategy |
|------|--------|----------|---------|
| EN→SA | English query | Sanskrit verse | Direct alignment |
| SA→EN | Sanskrit verse | English translation | Direct alignment |
| EN→EN | Synthetic query | English translation | Keyword-based synthetic |

**Hard negatives**: verses from different chapters (thematically distant).

In [ ]:
# ── Synthetic query generator ─────────────────────────────────────────────────
QUERY_TEMPLATES = [
    "What does this verse say about {}?",
    "Explain the concept of {} in the Gita.",
    "How does Krishna describe {}?",
    "What is the teaching on {} in this passage?",
    "Find verses about {}.",
]

TOPIC_KEYWORDS = [
    "karma", "dharma", "duty", "action", "self", "soul", "atman",
    "devotion", "knowledge", "meditation", "detachment", "yoga",
    "righteousness", "liberation", "moksha", "desire", "mind",
    "truth", "wisdom", "body", "renunciation", "sin", "God",
    "war", "fear", "faith", "peace", "senses", "creation",
]

def extract_keywords_from_text(text: str) -> list[str]:
    """Extract matching topic keywords from English text."""
    text_lower = text.lower()
    return [kw for kw in TOPIC_KEYWORDS if kw in text_lower]

def generate_query(text: str) -> str:
    """Generate a synthetic retrieval query for an English passage."""
    keywords = extract_keywords_from_text(text)
    if keywords:
        topic = random.choice(keywords)
    else:
        # Fallback: use first few content words
        words = [w for w in text.split() if len(w) > 4][:5]
        topic = random.choice(words) if words else "spiritual practice"
    return random.choice(QUERY_TEMPLATES).format(topic)

In [ ]:
# ── Hard Negative Sampler ─────────────────────────────────────────────────────
def get_hard_negative(idx: int, df: pd.DataFrame, use_chapter: bool = True) -> str:
    """
    Returns an English passage from a DIFFERENT chapter as a hard negative.
    Falls back to random if chapter info unavailable.
    """
    if use_chapter and 'chapter' in df.columns:
        current_chapter = df.iloc[idx]['chapter']
        diff_chapter = df[df['chapter'] != current_chapter]
        if len(diff_chapter) > 0:
            return diff_chapter.sample(1).iloc[0]['english']
    # Fallback: random row (not self)
    candidates = df.index[df.index != idx].tolist()
    return df.iloc[random.choice(candidates)]['english']

print("Example synthetic query:", generate_query(clean.iloc[0]['english']))
print("Example hard negative:\n", get_hard_negative(0, clean)[:120], "...")

In [ ]:
# ── Build InputExamples for MultipleNegativesRankingLoss ─────────────────────
# This loss expects (anchor, positive) pairs — negatives are other positives in the batch.
# We also include explicit hard negatives as (anchor, positive, hard_negative) triplets.

train_examples = []

for _, row in clean.iterrows():

    english = row["english"]
    sanskrit = row["sanskrit"]

    query = generate_query(english)

    train_examples.extend([

        InputExample(
            texts=[
                f"query: {query}",
                f"passage: {english}"
            ]
        ),

        InputExample(
            texts=[
                f"query: {query}",
                f"passage: {sanskrit}"
            ]
        ),

        InputExample(
            texts=[
                f"query: {english}",
                f"passage: {sanskrit}"
            ]
        ),

        InputExample(
            texts=[
                f"query: {sanskrit}",
                f"passage: {english}"
            ]
        )
    ])

# Shuffle
random.shuffle(train_examples)

# Train / val split
split = int(0.9 * len(train_examples))
train_data = train_examples[:split]
val_data   = train_examples[split:]

print(f"Total triplets  : {len(train_examples)}")
print(f"   Train           : {len(train_data)}")
print(f"   Val             : {len(val_data)}")
print(f"\nExample training pair:")

ex = train_data[0]

print(f"  Query   : {ex.texts[0][:120]}")
print(f"  Passage : {ex.texts[1][:120]}")

## 🤖 Section 4: Fine-Tune Embedding Model

**Model**: `intfloat/multilingual-e5-small`  
**Loss**: `TripletLoss` (margin-based) — ideal for explicit hard negatives  
**Alternative**: `MultipleNegativesRankingLoss` (in-batch negatives, faster)

In [ ]:
# BASE_MODEL = "Alibaba-NLP/gte-multilingual-base"   # ~120MB, multilingual
# OUTPUT_DIR = "./sanskrit_embedding_model"

# print(f"Loading base model: {BASE_MODEL}")
# model = SentenceTransformer(BASE_MODEL, device=DEVICE)
# print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

BASE_MODEL = "intfloat/multilingual-e5-small"
OUTPUT_DIR = "./sanskrit_embedding_model"

print(f"Loading base model: {BASE_MODEL}")

model = SentenceTransformer(
    BASE_MODEL,
    device=DEVICE
)

print(
    f"Embedding dimension: "
    f"{model.get_sentence_embedding_dimension()}"
)

In [ ]:
# ── Training configuration (fixed) ───────────────────────────────────────────
BATCH_SIZE   = 16        # smaller = more gradient updates, better for small data
EPOCHS       = 4         # 4 epoch only — dataset is tiny, 3 epochs = overfitting
WARMUP_STEPS = 100       # longer warmup stabilizes early training
LR           = 2e-5      # lower LR — prevents catastrophic forgetting

train_dataset = SentencesDataset(train_data, model)
train_loader  = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE)

# Switch to MultipleNegativesRankingLoss — much better for small datasets.
# It treats every other sample in the batch as a negative (in-batch negatives).
# No margin to tune, more stable than TripletLoss on small data.
from sentence_transformers import losses
loss_fn = losses.MultipleNegativesRankingLoss(model=model)

total_steps = len(train_loader) * EPOCHS
print(f"Steps per epoch : {len(train_loader)}")
print(f"Total steps     : {total_steps}")
print(f"Loss            : MultipleNegativesRankingLoss (in-batch negatives)")

In [ ]:
# ── Evaluator: Information Retrieval ─────────────────────────────────────────
# Build eval queries and corpus from val set
eval_queries   = {}   # qid -> query text
eval_corpus    = {}   # docid -> doc text
eval_relevant  = {}   # qid -> set of relevant docids

for i, ex in enumerate(val_data[:200]):   # limit eval size for speed
    qid   = f"q{i}"
    docid = f"d{i}"
    eval_queries[qid]  = ex.texts[0]   # anchor
    eval_corpus[docid] = ex.texts[1]   # positive
    eval_relevant[qid] = {docid}

ir_evaluator = evaluation.InformationRetrievalEvaluator(
    queries=eval_queries,
    corpus=eval_corpus,
    relevant_docs=eval_relevant,
    show_progress_bar=False,
    precision_recall_at_k=[1, 5, 10],
    mrr_at_k=[10],
    ndcg_at_k=[10],
    name="sanskrit_en_eval",
)

print("Evaluator ready.")
print(f"   Eval queries: {len(eval_queries)}, Corpus: {len(eval_corpus)}")

In [ ]:
# ── Evaluate BEFORE fine-tuning (baseline) ───────────────────────────────────
print("Baseline (pre-training) evaluation...")
baseline_score = ir_evaluator(model, output_path=None)

# ir_evaluator returns a dict of metric_name -> float
# Print all available keys so you can see the exact key names
print("Available metric keys:", list(baseline_score.keys()))

# Extract the primary MRR metric (key follows pattern: {name}_cosine_mrr@10)
MRR_KEY = next((k for k in baseline_score if 'mrr' in k.lower()), None)
NDCG_KEY = next((k for k in baseline_score if 'ndcg' in k.lower()), None)
R1_KEY   = next((k for k in baseline_score if 'precision' in k.lower() and '@1' in k), None)

print(f"\nBaseline MRR   : {baseline_score[MRR_KEY]:.4f}" if MRR_KEY else "MRR key not found")
print(f"Baseline nDCG  : {baseline_score[NDCG_KEY]:.4f}" if NDCG_KEY else "nDCG key not found")

In [ ]:
# ── Fine-Tune ─────────────────────────────────────────────────────────────────
print(f"Starting fine-tuning for {EPOCHS} epochs...")

model.fit(
    train_objectives=[(train_loader, loss_fn)],
    evaluator=ir_evaluator,
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    optimizer_params={'lr': LR},
    output_path=OUTPUT_DIR,
    save_best_model=True,
    show_progress_bar=True,
    evaluation_steps=len(train_loader),   # eval once per epoch
)

print(f"\nFine-tuning complete. Best model saved to: {OUTPUT_DIR}")

In [ ]:
# ── Load best model ───────────────────────────────────────────────────────────
best_model = SentenceTransformer(OUTPUT_DIR, device=DEVICE)
print("Best model loaded.")

## Section 5: Retrieval Evaluation — Recall@K, MRR, nDCG

In [ ]:
# ── Rebuild FAISS index with FINE-TUNED model (critical fix) ─────────────────
print("Building FAISS index with fine-tuned model...")

corpus_texts = []
corpus_meta  = []

for i, row in clean.iterrows():
    corpus_texts.append(row['english'])
    corpus_meta.append({'type': 'english', 'row': i})
    corpus_texts.append(row['sanskrit'])
    corpus_meta.append({'type': 'sanskrit', 'row': i})

def encode_passages(texts, model, prefix="passage: ", batch_size=64):
    prefixed = [prefix + t for t in texts]
    return model.encode(prefixed, batch_size=batch_size, show_progress_bar=True,
                        normalize_embeddings=True, convert_to_numpy=True)

def encode_queries(texts, model, prefix="query: ", batch_size=64):
    prefixed = [prefix + t for t in texts]
    return model.encode(prefixed, batch_size=batch_size, show_progress_bar=False,
                        normalize_embeddings=True, convert_to_numpy=True)

# Fine-tuned embeddings
ft_corpus_emb = encode_passages(corpus_texts, best_model)
dim = ft_corpus_emb.shape[1]

ft_index = faiss.IndexFlatIP(dim)
ft_index.add(ft_corpus_emb.astype('float32'))

# Baseline embeddings (same corpus, base model)
base_corpus_emb = encode_passages(corpus_texts, SentenceTransformer(BASE_MODEL, device=DEVICE))
base_index = faiss.IndexFlatIP(dim)
base_index.add(base_corpus_emb.astype('float32'))

print(f"Both indexes built. Vectors: {ft_index.ntotal}")

In [ ]:
# ── Build eval pairs with correct corpus index mapping ───────────────────────
def compute_retrieval_metrics(queries_with_gold, index, model, K_list=[1, 3, 5, 10]):
    """
    queries_with_gold: list of (query_text, gold_corpus_idx)
    gold_corpus_idx  : index into corpus_texts where the correct English doc lives
    """
    max_k  = max(K_list)
    q_vecs = encode_queries([q for q, _ in queries_with_gold], model)
    _, I   = index.search(q_vecs.astype('float32'), max_k)

    recall = defaultdict(list)
    mrr, ndcg = [], []

    for i, (_, gold_idx) in enumerate(queries_with_gold):
        retrieved = I[i].tolist()
        hit_ranks = [r+1 for r, doc in enumerate(retrieved) if doc == gold_idx]

        for k in K_list:
            recall[k].append(1 if any(doc == gold_idx for doc in retrieved[:k]) else 0)

        first_hit = hit_ranks[0] if hit_ranks else None
        mrr.append(1.0 / first_hit if first_hit else 0.0)

        gains = [1.0 if retrieved[j] == gold_idx else 0.0 for j in range(min(10, len(retrieved)))]
        ideal = sorted(gains, reverse=True)
        dcg   = sum(g / np.log2(j+2) for j, g in enumerate(gains))
        idcg  = sum(g / np.log2(j+2) for j, g in enumerate(ideal))
        ndcg.append(dcg / idcg if idcg > 0 else 0.0)

    results = {f"Recall@{k}": np.mean(recall[k]) for k in K_list}
    results["MRR@10"]  = np.mean(mrr)
    results["nDCG@10"] = np.mean(ndcg)
    return results

# Build eval pairs — gold_idx is position in corpus_texts (English = even slots: i*2)
clean_reset = clean.reset_index(drop=True)
sample_idx  = clean_reset.sample(min(200, len(clean_reset)), random_state=42).index.tolist()

eval_pairs = []
for i in sample_idx:
    query    = generate_query(clean_reset.iloc[i]['english'])
    gold_idx = i * 2   # English passages are at even positions in corpus_texts
    eval_pairs.append((query, gold_idx))

print(f"Eval pairs built: {len(eval_pairs)}")
print(f"   Sample gold_idx range: {min(g for _,g in eval_pairs)} – {max(g for _,g in eval_pairs)}")
print(f"   Corpus size: {len(corpus_texts)}")

# Sanity check
assert all(g < len(corpus_texts) for _, g in eval_pairs), "Gold index out of corpus range!"
print("Sanity check passed.")

In [ ]:
# ── Run evaluation on both models ────────────────────────────────────────────
print("Evaluating fine-tuned model...")
print("\nSample Evaluation Queries:")
for q, g in eval_pairs[:5]:
    print(f"Query: {q}")
    print(f"Gold Index: {g}")
    print()
metrics      = compute_retrieval_metrics(eval_pairs, ft_index,   best_model)

print("Evaluating baseline model...")
base_metrics = compute_retrieval_metrics(eval_pairs, base_index,
                                         SentenceTransformer(BASE_MODEL, device=DEVICE))

print("\n" + "="*50)
print(f"  {'Metric':<15} {'Baseline':>10} {'Fine-Tuned':>12} {'Δ':>8}")
print("  " + "-"*48)
for k in metrics:
    bv, fv = base_metrics[k], metrics[k]
    arrow  = "↑" if fv - bv > 0.001 else ("↓" if fv - bv < -0.001 else "=")
    print(f"  {k:<15} {bv:>10.4f} {fv:>12.4f}  {arrow}{abs(fv-bv):.4f}")

## Section 6: Embedding Space Analysis

In [ ]:
try:
    from sklearn.manifold import TSNE
    import matplotlib.cm as cm

    # Sample 100 verse pairs for visualization
    sample = clean.sample(min(100, len(clean)), random_state=42).reset_index(drop=True)

    en_texts  = ["passage: " + t for t in sample['english']]
    sa_texts  = ["passage: " + t for t in sample['sanskrit']]

    en_embs = best_model.encode(en_texts, normalize_embeddings=True)
    sa_embs = best_model.encode(sa_texts, normalize_embeddings=True)

    combined = np.vstack([en_embs, sa_embs])
    labels   = ['English'] * len(en_embs) + ['Sanskrit'] * len(sa_embs)
    colors   = ['steelblue'] * len(en_embs) + ['saddlebrown'] * len(sa_embs)

    tsne    = TSNE(n_components=2, random_state=42, perplexity=20)
    reduced = tsne.fit_transform(combined)

    plt.figure(figsize=(10, 8))
    for i, (x, y) in enumerate(reduced):
        plt.scatter(x, y, c=colors[i], alpha=0.7, s=40)
        # Draw lines connecting aligned pairs
        if i < len(en_embs):
            x2, y2 = reduced[i + len(en_embs)]
            plt.plot([x, x2], [y, y2], 'gray', alpha=0.15, linewidth=0.5)

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='steelblue', label='English'),
                       Patch(facecolor='saddlebrown', label='Sanskrit')]
    plt.legend(handles=legend_elements)
    plt.title('t-SNE: English vs Sanskrit Verse Embeddings (fine-tuned)\n(lines connect aligned pairs)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Compute mean cosine similarity between aligned pairs
    sim = np.sum(en_embs * sa_embs, axis=1)   # dot product of normalized = cosine
    print(f"\nMean cosine similarity (EN–SA aligned pairs): {sim.mean():.4f}")
    print(f"Min: {sim.min():.4f}  Max: {sim.max():.4f}")

except ImportError:
    print("sklearn not available — skipping t-SNE visualization")

## Section 7: Mini RAG Demo

In [ ]:
class SanskritRAG:
    """
    Retrieval-Augmented Generation system for Sanskrit–English queries.
    Retrieves relevant Gita verses and generates a synthesized answer.
    """

    def __init__(self, model, corpus_texts, corpus_meta, faiss_index, top_k=5):
        self.model        = model
        self.corpus_texts = corpus_texts
        self.corpus_meta  = corpus_meta
        self.index        = faiss_index
        self.top_k        = top_k

    def retrieve(self, query: str) -> list[dict]:
        """Retrieve top-k relevant verses."""
        q_vec = encode_queries([query], self.model)[0:1].astype('float32')
        scores, ids = self.index.search(q_vec, self.top_k)

        results = []
        seen_pairs = set()
        for score, idx in zip(scores[0], ids[0]):
            text = self.corpus_texts[idx]
            meta = self.corpus_meta[idx]
            row  = clean.iloc[meta['row']] if meta['row'] < len(clean) else None
            if row is not None:
                pair_key = meta['row']
                if pair_key not in seen_pairs:
                    seen_pairs.add(pair_key)
                    results.append({
                        'score'    : float(score),
                        'type'     : meta['type'],
                        'sanskrit' : row['sanskrit'],
                        'english'  : row['english'],
                        'chapter'  : row.get('chapter', 'N/A'),
                    })
        return results[:self.top_k]

    def summarize(self, query, results):

        answer = []

        answer.append(
            f"Question: {query}\n"
        )

        answer.append(
            "Based on the retrieved Bhagavad Gita verses:\n"
        )

        for r in results[:3]:

            answer.append(
                f"- {r['english'][:150]}"
            )

        return "\n".join(answer)

    def answer(self, query: str) -> str:
        """Retrieve + format a synthesized answer."""
        results = self.retrieve(query)
        if not results:
            return "No relevant passages found."

        lines = [f"Query: {query}\n",
                 "=" * 60,
                 f"Top {len(results)} relevant Bhagavad Gita passages:\n"]

        for i, r in enumerate(results, 1):
            lines.append(f"--- Passage {i} (Chapter {r['chapter']}, score: {r['score']:.3f}) ---")
            lines.append(f"🕉️  Sanskrit : {r['sanskrit'][:120]}..." if len(r['sanskrit']) > 120 else f"🕉️  Sanskrit : {r['sanskrit']}")
            lines.append(f"📖  English  : {r['english'][:200]}..." if len(r['english']) > 200 else f"📖  English  : {r['english']}")
            lines.append("")

            lines.append(
                "\nSynthesized Answer\n"
            )

            lines.append(
                self.summarize(
                    query,
                    results
                )
            )

        return "\n".join(lines)


# Build RAG system
rag = SanskritRAG(best_model, corpus_texts, corpus_meta, ft_index, top_k=3)
print("RAG system ready.")

In [ ]:
# ── Demo queries ──────────────────────────────────────────────────────────────
demo_queries = [
    "What does this verse say about karma?",
    "Explain the concept of dharma and righteous duty.",
    "कर्म के बारे में कृष्ण क्या कहते हैं?",   # Hindi query (cross-lingual)
    "What is the nature of the eternal soul?",
    "How should one deal with fear in battle?",
]

for q in demo_queries:
    print(rag.answer(q))
    print("\n" + "="*60 + "\n")

## Section 8: Transliteration Robustness Test

One key challenge is **transliteration mismatch** — the same Sanskrit word can appear as:
- Devanagari: `धर्म`
- IAST: `dharma`  
- Informal: `Dharma`, `dharm`

We test whether our model handles these variants.

In [ ]:

transliteration_tests = [

    ("Karma",
     "What is karma?",
     "What is कर्म?"),

    ("Dharma",
     "Explain dharma.",
     "Explain धर्म."),

    ("Atman",
     "What is atman?",
     "What is आत्मन्?"),

    ("Yoga",
     "What is yoga?",
     "What is योग?"),

    ("Moksha",
     "What is moksha?",
     "What is मोक्ष?"),

    ("Bhakti",
     "Explain bhakti.",
     "Explain भक्ति.")
]

print("\n")
print("="*60)
print("Cross-Lingual Similarity Examples")
print("="*60)

pairs = [

    (
        "What is karma?",
        "कर्म"
    ),

    (
        "What is yoga?",
        "योग"
    ),

    (
        "What is soul?",
        "आत्मा"
    )
]

for en, sa in pairs:

    v1 = encode_queries(
        [en],
        best_model
    )[0]

    v2 = encode_queries(
        [sa],
        best_model
    )[0]

    sim = np.dot(v1, v2)

    print(f"{en}")
    print(f"{sa}")
    print(f"Similarity: {sim:.4f}")
    print("-"*40)

print("Transliteration Robustness Test")
print("=" * 60)
print("Measuring cosine similarity between romanized and Devanagari queries.")
print("High similarity = model is robust to transliteration mismatch.\n")

for desc, q1, q2 in transliteration_tests:
    v1 = encode_queries([q1], best_model)[0]
    v2 = encode_queries([q2], best_model)[0]
    # Cosine similarity of normalized vectors = dot product
    sim = float(np.dot(v1, v2))
    print(f"  {desc:10s}: '{q1}' ↔ '{q2}'")
    print(f"             Cosine similarity: {sim:.4f} {'✅' if sim > 0.8 else '⚠️ (mismatch)'}\n")

## Section 9: Chunking Strategy Analysis

In [ ]:
def chunk_by_verse(df: pd.DataFrame) -> list[dict]:
    """1 chunk = 1 verse (current strategy)."""
    return [{'text': row['english'], 'chunk_type': 'verse',
             'size': len(row['english'])} for _, row in df.iterrows()]

def chunk_by_chapter(df: pd.DataFrame) -> list[dict]:
    """1 chunk = all verses in a chapter."""
    if 'chapter' not in df.columns:
        return chunk_by_verse(df)
    chunks = []
    for ch, grp in df.groupby('chapter'):
        text = ' '.join(grp['english'].tolist())
        chunks.append({'text': text, 'chunk_type': 'chapter',
                       'size': len(text), 'chapter': ch})
    return chunks

def chunk_sliding_window(df: pd.DataFrame, window: int = 3) -> list[dict]:
    """Sliding window of `window` consecutive verses."""
    texts  = df['english'].tolist()
    chunks = []
    for i in range(len(texts) - window + 1):
        text = ' '.join(texts[i:i+window])
        chunks.append({'text': text, 'chunk_type': f'window_{window}',
                       'size': len(text)})
    return chunks

strategies = {
    'verse'    : chunk_by_verse(clean),
    'chapter'  : chunk_by_chapter(clean),
    'window_3' : chunk_sliding_window(clean, 3),
    'window_5' : chunk_sliding_window(clean, 5),
}

print("\nChunking Strategy Comparison")
print("=" * 50)
print(f"{'Strategy':<15} {'# Chunks':>10} {'Avg Size (chars)':>18}")
print("-" * 50)
for name, chunks in strategies.items():
    avg_size = np.mean([c['size'] for c in chunks])
    print(f"{name:<15} {len(chunks):>10} {avg_size:>18.0f}")

## Section 10: Save & Export

In [ ]:
import pickle

# Save FAISS index
faiss.write_index(ft_index, "./sanskrit_faiss.index")

# Save corpus metadata
with open('./corpus_texts.pkl', 'wb') as f:
    pickle.dump({'texts': corpus_texts, 'meta': corpus_meta}, f)

# Save evaluation metrics
metrics_report = {
    'model'         : BASE_MODEL,
    'fine_tuned'    : True,
    'train_triplets': len(train_data),
    'corpus_size'   : len(corpus_texts),
    'metrics'       : metrics,
    'baseline'      : base_metrics,
}
with open('./eval_report.json', 'w') as f:
    json.dump(metrics_report, f, indent=2)

print("Saved:")
print("  ./sanskrit_embedding_model/  (fine-tuned model)")
print("  ./sanskrit_faiss.index       (FAISS retrieval index)")
print("  ./corpus_texts.pkl           (corpus metadata)")
print("  ./eval_report.json           (evaluation report)")

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Base model           : {BASE_MODEL}")
print(f"Training triplets    : {len(train_data)}")
print(f"Epochs               : {EPOCHS}")
print(f"Corpus size          : {len(corpus_texts)} passages ({len(clean)} verses × 2 langs)")
print()
# print("Retrieval Metrics (Fine-Tuned):")
print("\nRetrieval Performance")
if metrics["Recall@10"] > 0.30:
    print("\nRetrieval quality is good")
elif metrics["Recall@10"] > 0.15:
    print("\nRetrieval quality is moderate")
else:
    print("\nRetrieval quality needs improvement")
for k, v in metrics.items():
    base_v = base_metrics.get(k, 0)
    delta  = v - base_v
    arrow  = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(f"  {k:15s}: {v:.4f}  (baseline: {base_v:.4f}, {arrow}{abs(delta):.4f})")

In [ ]:
# ── Interactive Query Interface ───────────────────────────────────────────
# Professor can type any question here and get relevant Gita verses back

QUERY = "What does Krishna say about duty and action without attachment?"  # ← change this line

# ── Run retrieval ─────────────────────────────────────────────────────────────
query_vec = encode_queries([QUERY], best_model).astype('float32')
scores, ids = ft_index.search(query_vec, k=5)

print(f"\nQuery: {QUERY}")
print("=" * 70)

for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), 1):
    row_i   = idx // 2          # each verse occupies 2 slots (en, sa)
    lang    = "English" if idx % 2 == 0 else "Sanskrit"
    verse   = clean_reset.iloc[row_i]
    chapter = verse.get('chapter', 'N/A') if hasattr(verse, 'get') else verse['chapter'] if 'chapter' in verse else 'N/A'

    print(f"\n📌 Rank {rank}  |  Score: {score:.4f}  |  Chapter: {chapter}  |  Matched: {lang}")
    print(f"🕉️  Sanskrit : {verse['sanskrit']}")
    print(f"📖  English  : {verse['english']}")
    print("-" * 70)

In [ ]:
# ── Test with multiple questions (run this to demo the system) ─────────────

test_queries = [
    # English conceptual queries
    "What does this verse say about karma?",
    "How should a warrior overcome fear?",
    "What is the nature of the eternal soul?",
    "Explain the concept of detachment from results.",
    "What does Krishna say about meditation?",

    # Sanskrit keyword queries
    "योग और ध्यान के बारे में क्या कहा गया है?",   # Hindi: What is said about yoga and meditation?
    "धर्म का पालन कैसे करें?",                      # Hindi: How to follow dharma?

    # Direct Sanskrit verse fragment (copy-paste a partial verse)
    clean_reset.iloc[0]['sanskrit'][:40],            # first 40 chars of verse 1
]

for q in test_queries:
    q_vec = encode_queries([q], best_model).astype('float32')
    sc, ids = ft_index.search(q_vec, k=3)

    print(f"\n{q}")
    print(f"   → Top match (score {sc[0][0]:.3f}): {clean_reset.iloc[ids[0][0]//2]['english'][:100]}...")
    print()

In [ ]:
# ── Cross-Check: Verify a specific known verse is retrievable ──────────────
# Pick a verse you fetched from Postman API and verify the system retrieves it

# Paste the exact English from the API response here
KNOWN_ENGLISH = "Dhrishtaketu, chekitana and the valiant king of Kasi, Purujit and Kuntibhoja and Saibya, the best men."

# Paste the exact Sanskrit from the API response here
KNOWN_SANSKRIT = "धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् |पुरुजित्कुन्तिभोजश्च शैब्यश्च नरपुंगवः"

print("=" * 70)
print("CROSS-CHECK TEST")
print("=" * 70)

def retrieve_and_check(query, label, expected_english, top_k=10):
    q_vec = encode_queries([query], best_model).astype('float32')
    scores, ids = ft_index.search(q_vec, k=top_k)

    print(f"\nQuery [{label}]: {query[:80]}")
    print(f"   Expected English : {expected_english[:80]}...")

    found_at_rank = None
    for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), 1):
        row_i         = idx // 2
        retrieved_eng = clean_reset.iloc[row_i]['english']

        # Check if the expected verse appears in this result
        match = expected_english[:40].lower() in retrieved_eng.lower()
        if match and found_at_rank is None:
            found_at_rank = rank
            print(f"   FOUND at Rank {rank} | Score: {score:.4f}")
            print(f"   Retrieved: {retrieved_eng[:100]}...")

    if found_at_rank is None:
        print(f"   NOT FOUND in top-{top_k} results")
        print(f"   Top-1 was: {clean_reset.iloc[ids[0][0]//2]['english'][:100]}...")

    return found_at_rank


# Test 1: Query with the exact English text (should be Rank 1)
retrieve_and_check(
    query    = KNOWN_ENGLISH,
    label    = "Exact English",
    expected_english = KNOWN_ENGLISH
)

# Test 2: Query with the Sanskrit verse (cross-lingual)
retrieve_and_check(
    query    = KNOWN_SANSKRIT,
    label    = "Sanskrit → English",
    expected_english = KNOWN_ENGLISH
)

# Test 3: Query with a paraphrased question about the verse
retrieve_and_check(
    query    = "What does Krishna say about following one's own duty over another's?",
    label    = "Paraphrased Question",
    expected_english = KNOWN_ENGLISH
)

# Test 4: Keyword-only query
retrieve_and_check(
    query    = "own dharma imperfectly performed",
    label    = "Keyword Query",
    expected_english = KNOWN_ENGLISH
)

In [ ]:
import re

def parse_chapter_verse(query: str):
    """Extract chapter/verse numbers from query if mentioned."""
    chapter_match = re.search(
        r'chapter\s*(\d+)', query.lower()
    )
    verse_match = re.search(
        r'verse\s*(\d+)', query.lower()
    )
    chapter = int(chapter_match.group(1)) if chapter_match else None
    verse   = int(verse_match.group(1))   if verse_match   else None
    return chapter, verse

def filter_by_metadata(chapter=None, verse=None, top_k=3):
    """Directly filter corpus by chapter/verse instead of vector search."""
    results = []
    df = clean_reset.copy()

    if chapter and 'chapter' in df.columns:
        df = df[df['chapter'] == chapter]
    if verse and 'verse_num' in df.columns:
        df = df[df['verse_num'] == verse]

    for _, row in df.head(top_k).iterrows():
        results.append({
            'score'    : 1.0,   # exact match = perfect score
            'sanskrit' : row['sanskrit'],
            'english'  : row['english'],
            'chapter'  : row.get('chapter', 'N/A'),
            'verse_num': row.get('verse_num', 'N/A'),
        })
    return results

# Test
print(parse_chapter_verse("Explain Chapter 5"))
print(parse_chapter_verse("What does verse 4 say?"))
print(parse_chapter_verse("What is Chapter 3 verse 16 about?"))

In [ ]:
# Conversation memory
conversation_context = {
    'last_verses'  : [],    # last retrieved verses
    'last_query'   : None,
}

CONTEXT_TRIGGERS = [
    "this verse", "that verse", "it says", "explain this",
    "what does it mean", "elaborate", "tell me more", "expand"
]

def is_context_query(query: str) -> bool:
    """Check if user is referring to previously retrieved verse."""
    return any(trigger in query.lower() for trigger in CONTEXT_TRIGGERS)

def explain_last_verse(query):

    verse = conversation_context["last_verses"][0]

    explanation = f"""
📖 Explaining previously retrieved verse:

🕉️ Sanskrit:
{verse['sanskrit']}

📖 English:
{verse['english']}

📍 Chapter:
{verse['chapter']}

💡 Interpretation:

Krishna teaches that a person should focus on performing their duty sincerely without becoming attached to the results. Success and failure should not determine one's actions. The emphasis is on right action, self-discipline, and detachment from outcomes.
"""

    return explanation

In [ ]:
GREETINGS = {
    "hi",
    "hello",
    "hey",
    "namaste",
    "good morning",
    "good afternoon",
    "good evening",
    "hola",
    "greetings"
}

GITA_KEYWORDS = {
    "karma",
    "dharma",
    "yoga",
    "moksha",
    "krishna",
    "arjuna",
    "gita",
    "soul",
    "atma",
    "bhakti",
    "duty",
    "meditation",
    "self",
    "god",
    "spiritual",
    "कर्म",
    "धर्म",
    "योग",
    "मोक्ष",
    "आत्मा",
    "गीता",
    "कृष्ण",
    "अर्जुन"
}

In [ ]:
# ── Upgraded Chatbot — All Problems Fixed ─────────────────────────────

print("=" * 60)
print("🕉️  Bhagavad Gita RAG Chatbot (v2)")
print("=" * 60)
print("Supports: English | Sanskrit | Hindi")
print("Examples:")
print("  • What is karma?")
print("  • Explain Chapter 3")
print("  • What does verse 16 say?")
print("  • कर्मण्येवाधिकारस्ते")
print("  • (after a result) What does this verse mean?")
print("Type 'quit' to exit.\n")

RELEVANCE_THRESHOLD = 0.30


def is_greeting(text: str) -> bool:
    return text.lower().strip() in GREETINGS


def is_gita_related(text: str) -> bool:
    text_lower = text.lower()
    return any(kw in text_lower for kw in GITA_KEYWORDS)


def is_gibberish(text: str) -> bool:
    if len(text.strip()) < 3:
        return True

    alpha_ratio = sum(c.isalpha() for c in text) / len(text)

    if alpha_ratio < 0.4:
        return True

    return False


while True:

    try:
        query = input("You: ").strip()

    except (EOFError, KeyboardInterrupt):
        print("\nNamaste!")
        break

    if not query:
        continue

    # Exit and show evaluation metrics
    if query.lower() in ["quit", "exit", "bye", "q"]:

        print("\nBot: Namaste!")

        print("\n" + "=" * 60)
        print("FINAL RETRIEVAL EVALUATION")
        print("=" * 60)

        print(f"Recall@10 : {metrics['Recall@10']:.4f}")
        print(f"MRR@10    : {metrics['MRR@10']:.4f}")
        print(f"nDCG@10   : {metrics['nDCG@10']:.4f}")

        print("=" * 60)

        break

    if is_greeting(query):
        print("Bot: Namaste! Ask me anything about the Bhagavad Gita.\n")
        continue

    if is_gibberish(query):
        print("Bot: Please ask a meaningful question.\n")
        continue

    print("\nBot:")
    print("-" * 60)

    # ── Route 1: Follow-up Context Questions ────────────────────────────
    # if is_context_query(query) and conversation_context["last_verses"]:

    chapter_num, verse_num = parse_chapter_verse(query)

    if (
        is_context_query(query)
        and conversation_context["last_verses"]
        and not chapter_num
        and not verse_num
    ):

        print(explain_last_verse(query))

    # ── Route 2: Chapter / Verse Metadata Search ────────────────────────
    else:

        chapter_num, verse_num = parse_chapter_verse(query)

        if chapter_num or verse_num:

            results = filter_by_metadata(
                chapter_num,
                verse_num,
                top_k=3
            )

            if not results:

                label = (
                    f"Chapter {chapter_num}"
                    if chapter_num
                    else f"Verse {verse_num}"
                )

                print(f"No results found for {label}.\n")

            else:

                label = (
                    f"Chapter {chapter_num}"
                    if chapter_num
                    else f"Verse {verse_num}"
                )

                print(
                    f"Showing results for {label} "
                    f"({'exact match' if verse_num else 'all verses'}):\n"
                )

                for i, r in enumerate(results, 1):

                    print(
                        f"Result {i} | "
                        f"Chapter {r['chapter']} "
                        f"Verse {r['verse_num']}"
                    )

                    print(f"🕉️  Sanskrit : {r['sanskrit']}")
                    print(f"📖  English  : {r['english']}\n")

                conversation_context["last_verses"] = results

        # ── Route 3: Semantic Retrieval ─────────────────────────────────
        # else:

        #     q_vec = encode_queries(
        #         [query],
        #         best_model
        #     ).astype("float32")

        #     scores, ids = ft_index.search(q_vec, k=3)

        # ── Route 3: Semantic Retrieval ─────────────────────────────────
        else:

            # First check for exact Sanskrit verse match
            exact_match = clean_reset[
                clean_reset["sanskrit"].str.contains(
                    query,
                    regex=False,
                    na=False
                )
            ]

            if len(exact_match):

                row = exact_match.iloc[0]

                print(
                    f"Exact Match | "
                    f"Chapter {row['chapter']} "
                    f"Verse {row['verse_num']}"
                )

                print(f"🕉️ Sanskrit : {row['sanskrit']}")
                print(f"📖 English  : {row['english']}\n")

                conversation_context["last_verses"] = [{
                    "sanskrit": row["sanskrit"],
                    "english": row["english"],
                    "chapter": row["chapter"],
                    "verse_num": row["verse_num"],
                    "score": 1.0
                }]

                print("-" * 60 + "\n")
                continue

            # Otherwise perform FAISS semantic search
            q_vec = encode_queries(
                [query],
                best_model
            ).astype("float32")

            scores, ids = ft_index.search(q_vec, k=3)

            best_score = float(scores[0][0])
            print(f"DEBUG SCORE = {best_score:.4f}")

            # if best_score < RELEVANCE_THRESHOLD:
            is_sanskrit = any('\u0900' <= c <= '\u097F' for c in query)

            if best_score < RELEVANCE_THRESHOLD and not is_sanskrit:

                print(
                    f"No confident match found "
                    f"(score: {best_score:.3f})."
                )

                print(
                    "Try using keywords like:\n"
                    "karma, dharma, yoga, soul, Krishna\n"
                )

                print("-" * 60 + "\n")

                continue

            retrieved = []
            seen = set()

            for score, idx in zip(scores[0], ids[0]):

                row_i = idx // 2

                if row_i in seen:
                    continue

                seen.add(row_i)

                row = clean_reset.iloc[row_i]

                retrieved.append({
                    "score": float(score),
                    "sanskrit": row["sanskrit"],
                    "english": row["english"],
                    "chapter": row.get("chapter", "N/A"),
                    "verse_num": row.get("verse_num", "N/A"),
                })

            for i, r in enumerate(retrieved, 1):

                print(
                    f"📌 Result {i} | "
                    f"Chapter {r['chapter']} "
                    f"Verse {r['verse_num']} | "
                    f"Score: {r['score']:.3f}"
                )

                print(f"🕉️  Sanskrit : {r['sanskrit']}")
                print(f"📖  English  : {r['english']}\n")

            # Optional query confidence
            print("Query Analysis")

            if best_score >= 0.45:
                confidence = "High"
            elif best_score >= 0.35:
                confidence = "Medium"
            else:
                confidence = "Low"

            print(f"Best Match Score : {best_score:.3f}")
            print(f"Confidence       : {confidence}")
            print(f"Retrieved Verses : {len(retrieved)}")

            conversation_context["last_verses"] = retrieved
            conversation_context["last_query"] = query

    print("-" * 60 + "\n")